<a href="https://colab.research.google.com/github/ohNwghtEG/Multi-Asset-Portfolio-Performance-Risk-Dashboard/blob/main/Multi-Asset%20Portfolio%20Performance%20%26%20Risk%20Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ─────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────

In [2]:
!pip install yfinance fredapi plotly pandas numpy quantstats -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 1.8 MB/s eta 0:00:00


# ─────────────────────────────────────────
# CELL 2 — Set Up Your FRED API Key
# ─────────────────────────────────────────

In [3]:
from google.colab import userdata
FRED_KEY = userdata.get('FRED_KEY')
print("FRED key loaded:", FRED_KEY[:6] + "..." if FRED_KEY else "NOT FOUND")

FRED key loaded: 990420...


# ─────────────────────────────────────────
# CELL 3 — Imports & Configuration
# ─────────────────────────────────────────

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from fredapi import Fred
import warnings
warnings.filterwarnings('ignore')

# ── Portfolio Configuration ──────────────────────────────────
# You can change these tickers and weights to anything you want.
# Weights must sum to 1.0.

TICKERS = ['SPY', 'AGG', 'GLD', 'QQQ', 'BTC-USD']
NAMES   = {
    'SPY':    'US Large Cap Equities',
    'AGG':    'US Aggregate Bonds',
    'GLD':    'Gold',
    'QQQ':    'Nasdaq 100 (Tech)',
    'BTC-USD':'Bitcoin',
}
WEIGHTS = {
    'SPY':    0.40,
    'AGG':    0.25,
    'GLD':    0.15,
    'QQQ':    0.10,
    'BTC-USD':0.10,
}
START_DATE = '2018-01-01'
END_DATE   = '2024-12-31'
BENCHMARK  = 'SPY'

print("Configuration loaded ✓")
print(f"Portfolio: {list(NAMES.values())}")

Configuration loaded ✓
Portfolio: ['US Large Cap Equities', 'US Aggregate Bonds', 'Gold', 'Nasdaq 100 (Tech)', 'Bitcoin']


# ─────────────────────────────────────────
# CELL 3.5 — NEW Helper Cell
# ─────────────────────────────────────────


In [5]:
# ── Shared helpers ───────────────────────────────────────────
# Defined once here so no chart cell depends on any other chart cell.

COLORS = {
    'SPY':    '#1f77b4',
    'AGG':    '#ff7f0e',
    'GLD':    '#ffd700',
    'QQQ':    '#9467bd',
    'BTC-USD':'#d62728',
}

BASE_LAYOUT = dict(
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.15),
)

def cumulative(returns):
    '''Growth of $1. Expects SIMPLE returns.'''
    return (1 + returns).cumprod()

def drawdown(returns):
    '''Fractional drawdown from running peak. Expects SIMPLE returns.'''
    cum = cumulative(returns)
    return (cum - cum.cummax()) / cum.cummax()

def rolling_sharpe(returns, window=60, periods=252):
    '''Vectorised rolling annualized Sharpe (no rf). ~100x faster than .apply().'''
    mu = returns.rolling(window).mean()
    sd = returns.rolling(window).std()
    return (mu / sd.replace(0, np.nan)) * np.sqrt(periods)

def portfolio_return(simple_returns, weights):
    '''Daily-rebalanced portfolio return. Weights are renormalised to sum to 1.'''
    w = pd.Series(weights)
    w = w / w.sum()
    return simple_returns[w.index].dot(w).rename('Portfolio')

def cagr(returns):
    cum = cumulative(returns)
    years = (cum.index[-1] - cum.index[0]).days / 365.25
    return cum.iloc[-1] ** (1 / years) - 1

def styled(fig, title, ytitle=None, xtitle='Date', height=450, **kw):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18)),
        xaxis_title=xtitle, yaxis_title=ytitle, height=height,
        **BASE_LAYOUT, **kw,
    )
    return fig

print("Helpers loaded ✓")

Helpers loaded ✓


# ─────────────────────────────────────────
# CELL 4 — Download Price Data
# ─────────────────────────────────────────


In [6]:
print("Downloading price data from Yahoo Finance...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)['Close']
prices = raw.dropna()

print(f"✓ Downloaded {len(prices)} trading days × {len(prices.columns)} assets")
print(f"  Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"\nFirst few rows:")
display(prices.head())

✓ Downloaded 1760 trading days × 5 assets
  Date range: 2018-01-02 → 2024-12-30

First few rows:


Ticker,AGG,BTC-USD,GLD,QQQ,SPY
Date,,,,,
2018-01-02,84.685654,14982.099609,125.150002,150.057236,235.954269
2018-01-03,84.693359,15201.000000,124.820000,151.515305,237.446701
2018-01-04,84.639091,15599.200195,125.459999,151.780411,238.447510
2018-01-05,84.584816,17429.500000,125.330002,153.304733,240.036530
2018-01-08,84.561501,15170.099609,125.309998,153.901184,240.475464


# ─────────────────────────────────────────
# CELL 5 — UPDATED Compute Returns & Fetch Risk-Free Rate
# ─────────────────────────────────────────

In [7]:
# Simple returns: the correct basis for weighting and compounding.
simple_returns = prices.pct_change().dropna()

# Log returns: kept for distribution/statistical work only. NOT compounded.
log_returns = np.log(prices / prices.shift(1)).dropna()

# 3-Month T-Bill from FRED as the risk-free rate
fred = Fred(api_key=FRED_KEY)
rf_series = fred.get_series('DGS3MO', observation_start=START_DATE) / 100
rf_daily  = rf_series.reindex(simple_returns.index, method='ffill') / 252
rf_daily  = rf_daily.fillna(rf_daily.median())

# ✅ FIX: portfolio return is the weighted sum of SIMPLE returns.
portfolio_returns = portfolio_return(simple_returns, WEIGHTS)

print("✓ Returns computed")
print(f"  Risk-free rate (latest): {rf_series.iloc[-1]:.2%}")

✓ Returns computed
  Risk-free rate (latest): 3.89%


# ─────────────────────────────────────────
# CELL 6 — UPDATED Performance Metrics Function
# ─────────────────────────────────────────


In [8]:
def compute_metrics(returns_df, rf_daily, label_map=None):
    '''Annualized risk/return metrics per column. Expects SIMPLE returns.'''
    results = {}
    for col in returns_df.columns:
        r  = returns_df[col].dropna()
        rf = rf_daily.reindex(r.index).ffill().fillna(0)

        ann_ret = r.mean() * 252
        ann_vol = r.std()  * np.sqrt(252)
        sharpe  = (r - rf).mean() / r.std() * np.sqrt(252)

        downside_std = r[r < 0].std() * np.sqrt(252)
        sortino = ann_ret / downside_std if downside_std > 0 else np.nan

        max_dd = drawdown(r).min()
        calmar = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

        var_95  = np.percentile(r, 5)
        cvar_95 = r[r <= var_95].mean()

        name = label_map.get(col, col) if label_map else col
        results[name] = {
            'Ann. Return':    f"{ann_ret:+.1%}",
            'CAGR':           f"{cagr(r):+.1%}",
            'Ann. Volatility':f"{ann_vol:.1%}",
            'Sharpe Ratio':   f"{sharpe:.2f}",
            'Sortino Ratio':  f"{sortino:.2f}",
            'Max Drawdown':   f"{max_dd:.1%}",
            'Calmar Ratio':   f"{calmar:.2f}",
            'Daily VaR (95%)':f"{var_95:.2%}",
            'CVaR (95%)':     f"{cvar_95:.2%}",
        }
    return pd.DataFrame(results).T


# ✅ FIX: was log_returns
metrics = compute_metrics(simple_returns, rf_daily, label_map=NAMES)
print("=" * 60)
print("PORTFOLIO METRICS SUMMARY")
print("=" * 60)
display(metrics)

PORTFOLIO METRICS SUMMARY


,Ann. Return,CAGR,Ann. Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Calmar Ratio,Daily VaR (95%),CVaR (95%)
US Aggregate Bonds,+1.2%,+1.0%,6.0%,-0.21,0.24,-18.4%,0.06,-0.55%,-0.87%
Bitcoin,+49.2%,+29.8%,67.5%,0.69,0.98,-81.4%,0.60,-6.11%,-9.85%
Gold,+10.4%,+9.8%,14.3%,0.56,1.02,-22.0%,0.47,-1.48%,-2.10%
Nasdaq 100 (Tech),+20.5%,+19.2%,24.1%,0.75,1.10,-35.1%,0.58,-2.49%,-3.60%
US Large Cap Equities,+14.7%,+13.7%,19.5%,0.63,0.92,-33.7%,0.44,-1.85%,-3.00%


# ─────────────────────────────────────────
# CELL 7 — Chart 1: Cumulative Returns
# ─────────────────────────────────────────

In [9]:
fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Scatter(
        x=prices.index, y=cumulative(simple_returns[ticker]),
        name=NAMES[ticker], line=dict(width=2, color=COLORS[ticker]),
        hovertemplate=f'<b>{NAMES[ticker]}</b><br>Date: %{{x|%b %d, %Y}}<br>Growth of $1: $%{{y:.2f}}<extra></extra>'))
styled(fig, 'Cumulative Returns — Growth of $1 (2018–2024)', 'Growth of $1', height=500).show()

# ─────────────────────────────────────────
# CELL 8 — Chart 2: Drawdown (Underwater) Chart
# ─────────────────────────────────────────

In [10]:
fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Scatter(
        x=prices.index, y=drawdown(simple_returns[ticker]) * 100,
        name=NAMES[ticker], fill='tozeroy',
        line=dict(width=1.5, color=COLORS[ticker]),
        hovertemplate=f'<b>{NAMES[ticker]}</b><br>Date: %{{x|%b %d, %Y}}<br>Drawdown: %{{y:.1f}}%<extra></extra>'))
styled(fig, 'Drawdown — Underwater Chart', 'Drawdown (%)',
       yaxis=dict(ticksuffix='%')).show()

NameError: name 'colors' is not defined


# ─────────────────────────────────────────
# CELL 9 — Chart 3: Rolling 60-Day Sharpe Ratio
# ────────────────────────────────────────

In [ ]:
fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Scatter(
        x=prices.index, y=rolling_sharpe(simple_returns[ticker]),
        name=NAMES[ticker], line=dict(width=1.8, color=COLORS[ticker])))
fig.add_hline(y=0, line_dash='dash', line_color='red', line_width=1,
              annotation_text='Break-even', annotation_position='bottom right')
fig.add_hline(y=1, line_dash='dot', line_color='green', line_width=1,
              annotation_text='Sharpe = 1', annotation_position='top right')
styled(fig, 'Rolling 60-Day Annualized Sharpe Ratio', 'Sharpe Ratio (Annualized)').show()

# ─────────────────────────────────────────
# CELL 10 — Chart 4: Correlation Heatmap
# ─────────────────────────────────────────

In [14]:
corr_matrix = simple_returns.corr()
corr_matrix.index   = [NAMES[t] for t in corr_matrix.index]
corr_matrix.columns = [NAMES[t] for t in corr_matrix.columns]

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='Asset Return Correlation Matrix (Full Period)',
    aspect='auto',
)
fig.update_layout(
    template='plotly_white',
    height=450,
    coloraxis_colorbar=dict(title='Correlation'),
)
fig.show()

print("\nKey insight: Look for low/negative correlations (blue) —")
print("these are the diversification pairs that reduce portfolio risk.")


Key insight: Look for low/negative correlations (blue) —
these are the diversification pairs that reduce portfolio risk.



# ─────────────────────────────────────────
# CELL 11 — Chart 5: Return Distribution with VaR Lines
# ─────────────────────────────────────────

In [13]:
fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Histogram(
        x=simple_returns[ticker] * 100, name=NAMES[ticker],
        opacity=0.5, nbinsx=80, marker_color=COLORS[ticker]))
styled(fig, 'Daily Return Distributions', 'Frequency',
       xtitle='Daily Return (%)', barmode='overlay', hovermode='closest').show()

TypeError: plotly.graph_objs._figure.Figure.update_layout() got multiple values for keyword argument 'hovermode'

# ─────────────────────────────────────────
# CELL 12 — Chart 6: Bull/Bear Regime with 200-Day MA
# ─────────────────────────────────────────

In [11]:
spy_prices = prices['SPY']
ma_200 = spy_prices.rolling(200).mean()

# Classify regime
bull_mask = spy_prices > ma_200

fig = go.Figure()

# Background shading for bear regimes
bull_mask = prices['SPY'] > ma_200
segments  = (bull_mask != bull_mask.shift()).cumsum()

fig = go.Figure()
for _, seg in prices['SPY'].groupby(segments):
    if not bull_mask.loc[seg.index[0]]:
        fig.add_vrect(x0=seg.index[0], x1=seg.index[-1],
                      fillcolor='red', opacity=0.08, line_width=0)

fig.add_trace(go.Scatter(x=spy_prices.index, y=spy_prices,
                          name='SPY Price', line=dict(color='black', width=1.5)))
fig.add_trace(go.Scatter(x=ma_200.index, y=ma_200,
                          name='200-Day MA', line=dict(color='orange', width=2, dash='dash')))

fig.update_layout(
    title=dict(text='SPY Price with 200-Day MA Bull/Bear Regime (Red = Bear)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    legend=dict(orientation='h', y=-0.15),
    template='plotly_white',
    height=450,
)
fig.show()

# ─────────────────────────────────────────
# CELL 13 — Portfolio-Level Summary Dashboard (Combined)
# ─────────────────────────────────────────

In [17]:
# Compute portfolio cumulative returns
portfolio_cum = cumulative(portfolio_returns)
spy_cum       = cumulative(simple_returns['SPY'])

# Portfolio drawdown
rolling_max_port = portfolio_cum.cummax()
port_dd       = drawdown(portfolio_returns)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Portfolio vs SPY Benchmark — Cumulative Return',
        'Portfolio Drawdown',
        'Rolling 60-Day Portfolio Sharpe',
    ],
    vertical_spacing=0.07,
    row_heights=[0.5, 0.25, 0.25],
)

fig.add_trace(go.Scatter(x=portfolio_cum.index, y=portfolio_cum,
    name='Portfolio', line=dict(color='steelblue', width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=spy_cum.index, y=spy_cum,
    name='SPY Benchmark', line=dict(color='gray', width=1.5, dash='dash')), row=1, col=1)

fig.add_trace(go.Scatter(x=port_dd.index, y=port_dd * 100,
    name='Drawdown', fill='tozeroy', line=dict(color='red', width=1),
    fillcolor='rgba(255,0,0,0.15)'), row=2, col=1)

rolling_sharpe_port = rolling_sharpe(portfolio_returns)
fig.add_trace(go.Scatter(x=rolling_sharpe_port.index, y=rolling_sharpe_port,
    name='Rolling Sharpe', line=dict(color='green', width=1.5)), row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=3, col=1)

fig.update_layout(
    title=dict(text='Portfolio Dashboard (40% SPY / 25% AGG / 15% GLD / 10% QQQ / 10% BTC)', font=dict(size=16)),
    height=750,
    template='plotly_white',
    showlegend=True,
    legend=dict(orientation='h', y=-0.05),
)
fig.update_yaxes(title_text='Growth of $1', row=1, col=1)
fig.update_yaxes(title_text='Drawdown (%)', ticksuffix='%', row=2, col=1)
fig.update_yaxes(title_text='Sharpe Ratio', row=3, col=1)
fig.show()

# Print final summary stats
port_ann_ret = portfolio_returns.mean() * 252
port_ann_vol = portfolio_returns.std()  * np.sqrt(252)
port_sharpe  = port_ann_ret / port_ann_vol
port_max_dd  = port_dd.min()
print("\n" + "=" * 50)
print("PORTFOLIO SUMMARY (2018–2024)")
print("=" * 50)
print(f"  Annualized Return : {port_ann_ret:+.1%}")
print(f"  Annualized Vol    : {port_ann_vol:.1%}")
print(f"  Sharpe Ratio      : {port_sharpe:.2f}")
print(f"  Max Drawdown      : {port_max_dd:.1%}")
print(f"  Final Value ($1)  : ${portfolio_cum.iloc[-1]:.2f}")
print(f"  CAGR              : {cagr(portfolio_returns):+.1%}")


PORTFOLIO SUMMARY (2018–2024)
  Annualized Return : +14.7%
  Annualized Vol    : 14.5%
  Sharpe Ratio      : 1.02
  Max Drawdown      : -26.6%
  Final Value ($1)  : $2.60
  CAGR              : +14.6%
